# Learning Objectives

In this notebook, you will craft sophisticated ETL jobs that interface with a variety of common data sources, such as 
- REST APIs (HTTP endpoints)
- RDBMS
- Hive tables (managed tables)
- Various file formats (csv, json, parquet, etc.)

# Interview Questions

As you progress through the practice, attempt to answer the following questions:

## Columnar File
- What is a columnar file format and what advantages does it offer?
- Why is Parquet frequently used with Spark and how does it function?
- How do you read/write data from/to a Parquet file using a DataFrame?

## Partitions
- How do you save data to a file system by partitions? (Hint: Provide the code)
- How and why can partitions reduce query execution time? (Hint: Give an example)

## JDBC and RDBMS
- How do you load data from an RDBMS into Spark? (Hint: Discuss the steps and JDBC)

## REST API and HTTP Requests
- How can Spark be used to fetch data from a REST API? (Hint: Discuss making API requests)

## ETL Job One: Parquet file
### Extract
Extract data from the managed tables (e.g. `bookings_csv`, `members_csv`, and `facilities_csv`)

### Transform
Data transformation requirements https://pgexercises.com/questions/aggregates/fachoursbymonth.html

### Load
Load data into a parquet file

### What is Parquet? 

Columnar files are an important technique for optimizing Spark queries. Additionally, they are often tested in interviews.
- https://www.youtube.com/watch?v=KLFadWdomyI
- https://www.databricks.com/glossary/what-is-parquet

In [0]:
from pyspark.sql import functions as F

bks = spark.read.table("bookings")

result_df = (
    bks.filter((F.col("starttime") >= "2012-09-01") & (F.col("starttime") < "2012-10-01"))
    .groupBy("facid")
    .agg(F.sum("slots").alias("num_slots"))
    .orderBy(F.col("num_slots"))
)
result_df.show()

result_df.write.parquet("/delta/fachoursbymonth.parquet", mode="overwrite")


+-----+---------+
|facid|num_slots|
+-----+---------+
|    5|      122|
|    3|      422|
|    7|      426|
|    8|      471|
|    6|      540|
|    2|      570|
|    1|      588|
|    0|      591|
|    4|      648|
+-----+---------+



## ETL Job Two: Partitions

### Extract
Extract data from the managed tables (e.g. `bookings_csv`, `members_csv`, and `facilities_csv`)

### Transform
Transform the data https://pgexercises.com/questions/joins/threejoin.html

### Load
Partition the result data by facility column and then save to `threejoin_delta` managed table. Additionally, they are often tested in interviews.

hint: https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.DataFrameWriter.partitionBy.html

What are paritions? 

Partitions are an important technique to optimize Spark queries
- https://www.youtube.com/watch?v=hvF7tY2-L3U&t=268s

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import col

# extract data from managed tables
bookings_df = spark.read.table("bookings")
members_df = spark.read.table("members")
facilities_df = spark.read.table("facilities")

mbs = members_df.alias("mbs")
bks = bookings_df.alias("bks")
fcs = facilities_df.alias("fcs")

#transform data
result_df = (
    mbs.join(bks, col("mbs.memid") == col("bks.memid"))
    .join(fcs, col("bks.facid") == col("fcs.facid"))
    .filter(col("fcs.name").like("Tennis%"))
    .select(
        F.concat_ws(" ", col("mbs.firstname"), col("mbs.surname")).alias("member_name"),
        col("fcs.name").alias("court_name")
    )
    .distinct()
    .orderBy("member_name", "court_name")
)

result_df.show()

# partition by facility and save 
result_df.write.format("delta") \
    .partitionBy("court_name") \
    .mode("overwrite") \
    .saveAsTable("threejoin_delta")

+--------------+--------------+
|   member_name|    court_name|
+--------------+--------------+
|    Anne Baker|Tennis Court 1|
|    Anne Baker|Tennis Court 2|
|  Burton Tracy|Tennis Court 1|
|  Burton Tracy|Tennis Court 2|
|  Charles Owen|Tennis Court 1|
|  Charles Owen|Tennis Court 2|
|  Darren Smith|Tennis Court 2|
| David Farrell|Tennis Court 1|
| David Farrell|Tennis Court 2|
|   David Jones|Tennis Court 1|
|   David Jones|Tennis Court 2|
|  David Pinker|Tennis Court 1|
| Douglas Jones|Tennis Court 1|
| Erica Crumpet|Tennis Court 1|
|Florence Bader|Tennis Court 1|
|Florence Bader|Tennis Court 2|
|   GUEST GUEST|Tennis Court 1|
|   GUEST GUEST|Tennis Court 2|
|Gerald Butters|Tennis Court 1|
|Gerald Butters|Tennis Court 2|
+--------------+--------------+
only showing top 20 rows



## ETL Job Three: HTTP Requests

### Extract
Extract daily stock price data price from the following companies, Google, Apple, Microsoft, and Tesla. 

Data Source
- API: https://rapidapi.com/alphavantage/api/alpha-vantage
- Endpoint: GET `TIME_SERIES_DAILY`

Sample HTTP request

```
curl --request GET \
	--url 'https://alpha-vantage.p.rapidapi.com/query?function=TIME_SERIES_DAILY&symbol=TSLA&outputsize=compact&datatype=json' \
	--header 'X-RapidAPI-Host: alpha-vantage.p.rapidapi.com' \
	--header 'X-RapidAPI-Key: [YOUR_KEY]'

```

Sample Python HTTP request

```
import requests

url = "https://alpha-vantage.p.rapidapi.com/query"

querystring = {
    "function":"TIME_SERIES_DAILY",
    "symbol":"IBM",
    "datatype":"json",
    "outputsize":"compact"
}

headers = {
    "X-RapidAPI-Host": "alpha-vantage.p.rapidapi.com",
    "X-RapidAPI-Key": "[YOUR_KEY]"
}

response = requests.get(url, headers=headers, params=querystring)

data = response.json()

# Now 'data' contains the daily time series data for "IBM"
```

### Transform
Find **weekly** max closing price for each company.

hints: 
  - Use a `for-loop` to get stock data for each company
  - Use the spark `union` operation to concat all data into one DF
  - create a new `week` column from the data column
  - use `group by` to calcualte max closing price

### Load
- Partition `DF` by company
- Load the DF in to a managed table called, `max_closing_price_weekly`

In [0]:
import requests
from pyspark.sql.functions import col, weekofyear, max

# api details
url = "https://alpha-vantage.p.rapidapi.com/query"
headers = {
    'x-rapidapi-key': "e175cee112msh17ecdc27c6fccfdp119230jsna7d32f6ea426",
    'x-rapidapi-host': "alpha-vantage.p.rapidapi.com"
}

# stock symbols
companies = ["GOOGL", "AAPL", "MSFT", "TSLA"]

# results dataframe
result_df = None  

for company in companies:
    # extract stock data from vantage
    querystring = {
        "function": "TIME_SERIES_DAILY",
        "symbol": company,
        "datatype": "json",
        "outputsize": "compact"
    }
    
    response = requests.get(url, headers=headers, params=querystring)
    data = response.json()

    # create dataframe
    df = spark.createDataFrame([
            {"date": date, "company": company, "closing_price": float(values["4. close"])}
            for date, values in data.get("Time Series (Daily)", {}).items()
    ])
    
    # add a 'week' column and find max
    weekly_df = (
        df.withColumn("week", weekofyear(col("date")))
        .groupBy("company", "week").agg(max("closing_price").alias("max_closing_price")).orderBy("week")
    )

    #  merge to one dataframe
    if result_df is None:
        result_df = weekly_df 
    else:
        result_df = result_df.union(weekly_df)

# partition by company and save
result_df.write \
    .partitionBy("company") \
    .format("parquet") \
    .mode("overwrite") \
    .saveAsTable("max_closing_price_weekly")

result_df.show()


+-------+----+-----------------+
|company|week|max_closing_price|
+-------+----+-----------------+
|  GOOGL|   1|           191.79|
|  GOOGL|   2|           196.87|
|  GOOGL|   3|            196.0|
|  GOOGL|   4|           200.21|
|  GOOGL|   5|           204.02|
|  GOOGL|   6|           206.38|
|  GOOGL|   7|           186.47|
|  GOOGL|   8|           185.27|
|  GOOGL|   9|           179.25|
|  GOOGL|  10|           173.86|
|  GOOGL|  11|           167.11|
|  GOOGL|  12|           164.29|
|  GOOGL|  43|           165.27|
|  GOOGL|  44|           174.46|
|  GOOGL|  45|           180.75|
|  GOOGL|  46|           181.62|
|  GOOGL|  47|           178.12|
|  GOOGL|  48|           169.23|
|  GOOGL|  49|           174.71|
|  GOOGL|  50|            195.4|
+-------+----+-----------------+
only showing top 20 rows



## ETL Job Four: RDBMS


### Extract
Extract RNA data from a public PostgreSQL database.

- https://rnacentral.org/help/public-database
- Extract 100 RNA records from the `rna` table (hint: use `limit` in your sql)
- hint: use `spark.read.jdbc` https://docs.databricks.com/external-data/jdbc.html

### Transform
We want to load the data as it so there is no transformation required.


### Load
Load the DF in to a managed table called, `rna_100_records`

In [0]:
# parameters
url = "jdbc:postgresql://hh-pgsql-public.ebi.ac.uk:5432/pfmegrnargs"
query = "(SELECT * FROM rna LIMIT 100) AS rna_data"
properties = {
    "user": "reader",
    "password": "NWDMCE5xdipIjRrp",
    "driver": "org.postgresql.Driver"
}

# extract records
rna_df = spark.read.jdbc(url=url, table=query, properties=properties)

# load the data into a managed table
rna_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("rna_100_records")

# verification
spark.sql("SELECT * FROM rna_100_records").show()


+--------+-------------+--------------------+---------+----------------+----+--------------------+--------+--------------------+
|      id|          upi|           timestamp|userstamp|           crc64| len|           seq_short|seq_long|                 md5|
+--------+-------------+--------------------+---------+----------------+----+--------------------+--------+--------------------+
| 9037999|URS000089E8AF| 2015-10-20 18:04:07|   RNACEN|F9D35C2B4B159BC2|1461|TGAACGCTGGCGGCAGG...|    null|247d52e4412d1dd22...|
| 9038003|URS000089E8B3| 2015-10-20 18:04:07|   RNACEN|414EA6C93A730A05|1453|TGGCTCAGGACGAACGC...|    null|247e2b651be18697d...|
| 9037948|URS000089E87C| 2015-10-20 18:04:07|   RNACEN|999DC5D0EF443188|1513|TTAGAGTTTGATCCTGG...|    null|247488a4c572ac05d...|
| 9037950|URS000089E87E| 2015-10-20 18:04:07|   RNACEN|D14B5F7D1E0786DD|1375|AGTCGCGCGAGAAATCC...|    null|7ec512659eb4a5d50...|
| 9037952|URS000089E880| 2015-10-20 18:04:07|   RNACEN|1792F1A724D8DBE2|1486|TGGCTCAGATTGAACGC...